### Opens a dataset and computes an average

In [ ]:
from distributed import Client
from prov_tracking import ProvTracker
import xarray as xr

#### Register Dask provenance tracker

In [ ]:
client: Client = Client()
plugin = ProvTracker(
  destination = '../../output',
  file_name = 'dataset_mean_jt.json',
  keep_traceback=True, rich_types=True,
  jupyter_tracking=True
)
client.register_plugin(plugin)
plugin.start(client.scheduler)
client

2025-09-29 15:10:42,551 - yprov4wfs - DEBUG - Using selector: EpollSelector
2025-09-29 15:10:44,418 - yprov4wfs - DEBUG - Initialized Workflow with id=448a4b94-086f-497d-835f-e43cfb09acf5, name=prov_tracking.plugin


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 23.20 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36691,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34887,Total threads: 2
Dashboard: http://127.0.0.1:45705/status,Memory: 5.80 GiB
Nanny: tcp://127.0.0.1:42133,


4: persist()
5: compute()
Binding ('open_dataset-air-3f1df329a711599ff3e0d046481d61be', 0, 0, 2) to 4: persist()


2025-09-29 15:10:46,373 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc3d3530>
2025-09-29 15:10:46,375 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc321880>
2025-09-29 15:10:46,376 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc3d0740>


Binding finalize-hlgfinalizecompute-d6a1e65cd1fd4ef3943d68d19f63ab4c to 5: compute()


2025-09-29 15:10:46,609 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc3d2390>
2025-09-29 15:10:46,609 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc3d34d0>
2025-09-29 15:10:46,619 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc3d3c80>
2025-09-29 15:10:47,227 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc3d3f80>
2025-09-29 15:10:47,228 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc0b1400>
2025-09-29 15:10:47,230 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc0b0d10>
2025-09-29 15:10:47,231 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc0b1490>
2025-09-29 15:10:47,249 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task object at 0x7f0ecc3d23f0>
2025-09-29 15:10:47,255 - yprov4wfs - INFO - Added task: <yprov4wfs.datamodel.task.Task 

#### Opens the dataset

In [3]:
ds = xr.tutorial.open_dataset(
  "air_temperature",
  chunks={  # this tells xarray to open the dataset as a dask array
    "lat": 25,
    "lon": 25,
    "time": -1,
  },
)
ds

<xarray.Dataset> Size: 31MB
Dimensions:  (lat: 25, time: 2920, lon: 53)
Coordinates:
  * lat      (lat) float32 100B 75.0 72.5 70.0 67.5 65.0 ... 22.5 20.0 17.5 15.0
  * lon      (lon) float32 212B 200.0 202.5 205.0 207.5 ... 325.0 327.5 330.0
  * time     (time) datetime64[ns] 23kB 2013-01-01 ... 2014-12-31T18:00:00
Data variables:
    air      (time, lat, lon) float64 31MB dask.array<chunksize=(2920, 25, 25), meta=np.ndarray>
Attributes:
    Conventions:  COARDS
    title:        4x daily NMC reanalysis (1948)
    description:  Data is from NMC initialized reanalysis\n(4x/day).  These a...
    platform:     Model
    references:   http://www.esrl.noaa.gov/psd/data/gridded/data.ncep.reanaly...

#### Gets the data

In [4]:
air = ds['air'].persist()
air

<xarray.DataArray 'air' (time: 2920, lat: 25, lon: 53)> Size: 31MB
dask.array<open_dataset-air, shape=(2920, 25, 53), dtype=float64, chunksize=(2920, 25, 25), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 100B 75.0 72.5 70.0 67.5 65.0 ... 22.5 20.0 17.5 15.0
  * lon      (lon) float32 212B 200.0 202.5 205.0 207.5 ... 325.0 327.5 330.0
  * time     (time) datetime64[ns] 23kB 2013-01-01 ... 2014-12-31T18:00:00
Attributes:
    long_name:     4xDaily Air temperature at sigma level 995
    units:         degK
    precision:     2
    GRIB_id:       11
    GRIB_name:     TMP
    var_desc:      Air temperature
    dataset:       NMC Reanalysis
    level_desc:    Surface
    statistic:     Individual Obs
    parent_stat:   Other
    actual_range:  [185.16 322.1 ]

#### Computes the average

In [5]:
ds_mean = air.mean(dim = 'time')
ds_mean = ds_mean.compute()

#### Closes the client and generates the provenance graph

In [6]:
client.shutdown()